[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Diffusion Models - Visual Intuition

Explore how data becomes noise and how repeated denoising generates samples without returning their average.

We use small Gaussian mixtures with a known density. The denoiser is computed analytically, so no network training, data download, or GPU is needed.

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        | Content / Changes |
|---------|------------|-------------|-------------------|
| 1.0.000 | 13/09/2026 | Royi Avital | Diffusion intuition and analytic visualizations |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2026_02/DiffusionModel.ipynb)

In [ ]:
# Import Packages

# Standard Library
import math
import os

# Scientific Python
import numpy as np
import scipy as sp

# Visualization
import matplotlib.pyplot as plt

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to explore the visualization.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code uses `vVector` for a vector, `mMatrix` for a matrix, `tTensor` for a tensor, and `oObj` for an object.

* <font color='brown'>(**#**)</font> Set `exportFig = True` to export the figures. Transparent backgrounds and 150 DPI match the slide plots; set `exportTransparent = False` to include the dark background.

In [ ]:
# Configuration

seedNum = 512
plt.style.use('dark_background')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'legend.fontsize': 10})

mColors = plt.get_cmap('tab10')(np.arange(10))
densityColor = 'cyan'
meanColor = 'white'

def FormatAxes( hA ) -> None:
    hA.grid(True, color = 'white', alpha = 0.12)
    hA.set_axisbelow(True)

def ShowFigure( hF, fileName: str ) -> None:
    hF.set_layout_engine('constrained', h_pad = 0.12, w_pad = 0.12)
    if exportFig:
        os.makedirs(exportDir, exist_ok = True)
        hF.savefig(os.path.join(exportDir, f'{fileName}.png'), dpi = 150, transparent = exportTransparent, bbox_inches = 'tight', pad_inches = 0.05)
    plt.show()
    plt.close(hF)

In [ ]:
# Parameters

# Diffusion
numDiffSteps = 400
numSamples = 8000

# One Dimensional Mixture
tuMeans = (-2.0, 2.0)
tuStds = (0.2, 0.2)
tuWeights = (0.5, 0.5)

# Visualization
numPlotSteps = 5
numTracks = 48
axisLimit = 4.0
numGridPts = 800
exportFig = False
exportTransparent = True
exportDir = os.path.join('Figures', 'DiffusionModel')

In [ ]:
# Diffusion Schedule

def CreateDiffusionSchedule( numSteps: int ):
    if numSteps < 2:
        raise ValueError('Use at least two diffusion steps.')
    vGrid = np.linspace(0, 1, numSteps + 1)
    vCurve = np.cos((vGrid + 0.008) / 1.008 * math.pi / 2) ** 2
    vBeta = np.clip(1 - vCurve[1:] / vCurve[:-1], 1e-5, 0.999)
    vAlpha = 1 - vBeta
    vAlphaBar = np.concatenate(([1.0], np.cumprod(vAlpha)))
    vPrev, vCurrent = vAlphaBar[:-1], vAlphaBar[1:]
    return {
        'AlphaBar': vAlphaBar,
        'Beta': vBeta,
        'CoefClean': vBeta * np.sqrt(vPrev) / (1 - vCurrent),
        'CoefNoisy': np.sqrt(vAlpha) * (1 - vPrev) / (1 - vCurrent),
        'Variance': vBeta * (1 - vPrev) / (1 - vCurrent),
    }

dSchedule = CreateDiffusionSchedule(numDiffSteps)

In [ ]:
# Analytic Gaussian Mixture

class GaussianMixture1D:
    def __init__( self, tuMeans, tuStds, tuWeights ) -> None:
        self.vMeans = np.asarray(tuMeans, dtype = float)
        self.vVars = np.asarray(tuStds, dtype = float) ** 2
        self.vWeights = np.asarray(tuWeights, dtype = float)
        if not (self.vMeans.shape == self.vVars.shape == self.vWeights.shape):
            raise ValueError('Means, standard deviations, and weights must have matching shapes.')
        if np.any(np.asarray(tuStds) <= 0) or np.any(self.vWeights <= 0):
            raise ValueError('Standard deviations and weights must be positive.')
        self.vWeights = self.vWeights / self.vWeights.sum()

    def LogComponents( self, vValue, alphaBar: float = 1.0 ):
        vValue = np.asarray(vValue, dtype = float).reshape(-1, 1)
        vNoisyVars = alphaBar * self.vVars + (1 - alphaBar)
        return np.log(self.vWeights) + sp.stats.norm.logpdf(vValue, loc = math.sqrt(alphaBar) * self.vMeans, scale = np.sqrt(vNoisyVars))

    def Density( self, vValue, alphaBar: float = 1.0 ):
        return np.exp(sp.special.logsumexp(self.LogComponents(vValue, alphaBar), axis = 1))

    def Denoise( self, vValue, alphaBar: float ):
        vValue = np.asarray(vValue, dtype = float).reshape(-1, 1)
        mLogProb = self.LogComponents(vValue, alphaBar)
        mProb = np.exp(mLogProb - sp.special.logsumexp(mLogProb, axis = 1, keepdims = True))
        vNoisyVars = alphaBar * self.vVars + (1 - alphaBar)
        vGain = math.sqrt(alphaBar) * self.vVars / vNoisyVars
        mMeans = self.vMeans + vGain * (vValue - math.sqrt(alphaBar) * self.vMeans)
        return (mProb * mMeans).sum(axis = 1)

    def Sample( self, numSamples: int, oRng ):
        vLabels = oRng.choice(len(self.vWeights), size = numSamples, p = self.vWeights)
        return self.vMeans[vLabels] + np.sqrt(self.vVars[vLabels]) * oRng.standard_normal(numSamples)

## From Data to Noise

A _Diffusion Model_ uses a fixed forward process and learns its reverse. Each forward step shrinks the signal and adds independent Gaussian noise:

$$ \boldsymbol{y}_t = \sqrt{\alpha_t} \boldsymbol{y}_{t-1} + \sqrt{1 - \alpha_t} \boldsymbol{\epsilon}_t, \qquad \boldsymbol{\epsilon}_t \sim \mathcal{N} \left( \boldsymbol{0}, \boldsymbol{I} \right) $$

The accumulated signal retention is $\bar\alpha_t = \prod_{j=1}^{t}\alpha_j$. We can also sample a noisy state directly:

$$ \boldsymbol{y}_t = \sqrt{\bar\alpha_t} \boldsymbol{y}_0 + \sqrt{1 - \bar\alpha_t} \boldsymbol{\epsilon} $$

As $\bar\alpha_t$ approaches zero, the distribution approaches standard Gaussian noise. Below, six groups merge as the same points pass through consecutive forward steps.

* <font color='brown'>(**#**)</font> Index `0` is clean data; index `numDiffSteps` is nearly pure noise. Colors track the original groups and are not supplied to a denoiser.
* <font color='red'>(**?**)</font> Why shrink the signal instead of only adding more noise?
* <font color='blue'>(**!**)</font> Change `clusterStd` or `clusterRadius`. Does the final distribution still approach standard Gaussian noise?

In [ ]:
# Forward Diffusion of Two Dimensional Clusters

clusterRadius = 4.0
clusterStd = 0.25
numClusters = 6
numPoints = 1800
oRng = np.random.default_rng(seedNum)

vAngles = np.linspace(0, 2 * math.pi, numClusters, endpoint = False)
mCenters = clusterRadius * np.column_stack((np.cos(vAngles), np.sin(vAngles)))
vLabels = oRng.integers(numClusters, size = numPoints)
mPoints = mCenters[vLabels] + clusterStd * oRng.standard_normal((numPoints, 2))
vShowSteps = np.linspace(0, numDiffSteps, numPlotSteps, dtype = int)
lSnapshots = [mPoints.copy()]
for stepIdx, betaVal in enumerate(dSchedule['Beta'], start = 1):
    mPoints = math.sqrt(1 - betaVal) * mPoints + math.sqrt(betaVal) * oRng.standard_normal(mPoints.shape)
    if stepIdx in vShowSteps[1:]:
        lSnapshots.append(mPoints.copy())

plotLimit = max(6.0, clusterRadius + 4 * clusterStd)
vCircle = np.linspace(0, 2 * math.pi, 300)
hF, vHA = plt.subplots(1, numPlotSteps, figsize = (4 * numPlotSteps, 5), sharex = True, sharey = True)
for stepIdx, mSnapshot, hA in zip(vShowSteps, lSnapshots, vHA):
    hA.scatter(mSnapshot[:, 0], mSnapshot[:, 1], c = mColors[vLabels], s = 5, alpha = 0.5, linewidths = 0)
    hA.plot(2 * np.cos(vCircle), 2 * np.sin(vCircle), '--', color = 'gray', linewidth = 1.5)
    hA.set(xlim = (-plotLimit, plotLimit), ylim = (-plotLimit, plotLimit), aspect = 'equal', xlabel = r'$y_1$', title = f't / T = {stepIdx / numDiffSteps:.2f}')
    hA.set_xticks((-4, 0, 4))
    hA.set_yticks((-4, 0, 4))
    FormatAxes(hA)
vHA[0].set_ylabel(r'$y_2$')
hF.suptitle('Forward Diffusion | Data to Noise', fontsize = 16)
ShowFigure(hF, '01ForwardClusters')

## The Mean Is Not a Typical Sample

Consider a balanced mixture of two narrow Gaussians:

$$ p(y_0) = \frac{1}{2} \mathcal{N} \left( y_0; -2, 0.2^2 \right) + \frac{1}{2} \mathcal{N} \left( y_0; 2, 0.2^2 \right) $$

Most samples lie near $-2$ or $2$, yet the mean is $\mathbb{E}[y_0] = 0$. Returning that mean every time would put all outputs in the low-density gap.

A _Mode_ is a local density maximum. A _MAP Estimate_ selects a global maximum; a _Sample_ is a random draw from the whole distribution.

* <font color='red'>(**?**)</font> What fraction of samples should come from each group? Must a sample be exactly at a density peak?

In [ ]:
# Bimodal Data Density

oMixture = GaussianMixture1D(tuMeans, tuStds, tuWeights)
vGrid = np.linspace(-axisLimit, axisLimit, numGridPts)
dataMean = float(oMixture.vWeights @ oMixture.vMeans)
vData = oMixture.Sample(numSamples, np.random.default_rng(seedNum + 1))

hF, hA = plt.subplots(figsize = (9, 4.5))
for compIdx, vComponent in enumerate(np.exp(oMixture.LogComponents(vGrid)).T):
    hA.fill_between(vGrid, vComponent, color = mColors[compIdx], alpha = 0.3)
hA.plot(vGrid, oMixture.Density(vGrid), color = densityColor, linewidth = 2.5, label = 'Data density')
hA.axvline(dataMean, color = meanColor, linestyle = '--', linewidth = 2, label = f'Mean = {dataMean:.1f}')
hA.scatter(vData[:200], np.full(200, -0.035), color = densityColor, marker = '|', alpha = 0.5, label = 'Samples')
hA.set(xlim = (-axisLimit, axisLimit), ylim = (-0.07, 1.2 * oMixture.Density(vGrid).max()), xlabel = r'$y_0$', ylabel = 'Density', title = 'Two Groups, One Mean')
hA.legend(loc = 'upper center')
FormatAxes(hA)
ShowFigure(hF, '02MeanAndSamples')

### Watch the Density Change

For component $k$, forward diffusion gives:

$$ y_t \mid k \sim \mathcal{N} \left( \sqrt{\bar\alpha_t}\mu_k,\; \bar\alpha_t\sigma_k^2 + 1 - \bar\alpha_t \right) $$

The centers move toward zero and the variances approach one. The cyan curve is the exact mixture density at each noise level; the dashed curve is the standard normal reference.

* <font color='red'>(**?**)</font> Is the average close to zero only at the final step, or throughout this balanced example?

In [ ]:
# Forward Marginal Densities

hF, vHA = plt.subplots(1, numPlotSteps, figsize = (4 * numPlotSteps, 3.8), sharex = True, sharey = True)
for stepIdx, hA in zip(vShowSteps, vHA):
    alphaBar = dSchedule['AlphaBar'][stepIdx]
    hA.plot(vGrid, oMixture.Density(vGrid, alphaBar), color = densityColor, linewidth = 2.5, label = r'$p(y_t)$')
    hA.plot(vGrid, sp.stats.norm.pdf(vGrid), '--', color = 'gray', linewidth = 1.5, label = r'$\mathcal{N}(0,1)$')
    hA.set(xlim = (-axisLimit, axisLimit), ylim = (0, 1.15 * oMixture.Density(vGrid).max()), xlabel = r'$y_t$', title = f't / T = {stepIdx / numDiffSteps:.2f}')
    FormatAxes(hA)
vHA[0].set_ylabel('Density')
vHA[-1].legend()
hF.suptitle('Forward Diffusion | Two Peaks Become One', fontsize = 16)
ShowFigure(hF, '03ForwardDensities')

## What Does an MSE Denoiser Return?

For a fixed noisy observation, the best clean prediction under _Mean Squared Error_ is a conditional mean:

$$ D^*(y_t,t) = \underset{d}{\operatorname{argmin}}\; \mathbb{E} \left[ (d-y_0)^2 \mid y_t,t \right] = \mathbb{E}[y_0 \mid y_t,t] $$

This is not the unconditional mean $\mathbb{E}[y_0]$. Different noisy observations give different posterior distributions and therefore different averages.

For our Gaussian mixture, Bayes' rule gives that posterior analytically. `Denoise()` averages the component-wise posterior means using their posterior probabilities; no network needs to be trained.

The left panel shows the denoiser at several signal levels. The right panel holds the noise level fixed and compares two observations. Dashed vertical lines mark their conditional means.

* <font color='red'>(**?**)</font> Why is the denoiser zero at $y_t=0$ in this balanced example? Does that mean it always returns zero?
* <font color='blue'>(**!**)</font> Change `posteriorAlphaBar` and `tuObservations`. When does the observation identify a group reliably?

In [ ]:
# Conditional Mean and Posterior Density

tuAlphaBars = (0.02, 0.25, 0.8)
posteriorAlphaBar = 0.15
tuObservations = (0.0, 1.0)

hF, vHA = plt.subplots(1, 2, figsize = (13, 4.8))
for curveIdx, alphaBar in enumerate(tuAlphaBars):
    vEstimate = oMixture.Denoise(vGrid, alphaBar)
    vHA[0].plot(vGrid, vEstimate, color = mColors[curveIdx], linewidth = 2.5, label = rf'$\bar\alpha_t = {alphaBar}$')
vHA[0].axhline(dataMean, color = meanColor, linestyle = '--', linewidth = 1.5, label = 'Unconditional mean')
vHA[0].plot(vGrid, vGrid, ':', color = 'gray', label = 'Identity')
vHA[0].set(xlim = (-axisLimit, axisLimit), ylim = (-3, 3), xlabel = r'Noisy observation $y_t$', ylabel = r'$\mathbb{E}[y_0 \mid y_t,t]$', title = 'One Denoiser, Different Observations')
vHA[0].legend(loc = 'upper left')

for obsIdx, observation in enumerate(tuObservations):
    vLikelihood = sp.stats.norm.pdf(observation, loc = math.sqrt(posteriorAlphaBar) * vGrid, scale = math.sqrt(1 - posteriorAlphaBar))
    vPosterior = oMixture.Density(vGrid) * vLikelihood / oMixture.Density([observation], posteriorAlphaBar)[0]
    posteriorMean = oMixture.Denoise([observation], posteriorAlphaBar)[0]
    vHA[1].plot(vGrid, vPosterior, color = mColors[obsIdx], linewidth = 2.5, label = rf'$y_t = {observation:.1f}$')
    vHA[1].axvline(posteriorMean, color = mColors[obsIdx], linestyle = '--', linewidth = 1.5)
vHA[1].set(xlim = (-axisLimit, axisLimit), ylim = (0, None), xlabel = r'Possible clean value $y_0$', ylabel = r'$p(y_0 \mid y_t,t)$', title = rf'Posterior at $\bar\alpha_t = {posteriorAlphaBar}$')
vHA[1].legend()
for hA in vHA:
    FormatAxes(hA)
ShowFigure(hF, '04ConditionalMean')

## Conditional Means Can Generate Diverse Samples

Start each trajectory at an independent Gaussian draw. At every reverse step, evaluate the denoiser at that trajectory's current state:

$$ \hat y_0 = D^*(y_t,t), \qquad y_{t-1} = c_{0,t}\hat y_0 + c_{t,t}y_t + \sqrt{\tilde\beta_t}\,z_t, \qquad z_t \sim \mathcal{N}(0,1) $$

The coefficients come from the known forward process:

$$ c_{0,t} = \frac{\beta_t\sqrt{\bar\alpha_{t-1}}}{1-\bar\alpha_t}, \qquad c_{t,t} = \frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}, \qquad \tilde\beta_t = \frac{\beta_t(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}, \qquad \beta_t=1-\alpha_t $$

* Different trajectories provide different observations to the same denoiser.
* As noise decreases, those observations increasingly identify one group.
* The average across trajectories can stay near zero while individual samples populate both groups.

* <font color='brown'>(**#**)</font> The denoiser is exact, but the finite-step Gaussian DDPM reverse transition is an approximation. The true reverse conditional is generally a mixture. Initialization is approximately Gaussian, and no extra noise is added at the final step.
* <font color='red'>(**?**)</font> Why does a conditional mean at each step not force all final samples to the unconditional mean?

In [ ]:
# Analytic Denoiser in a DDPM Sampler

def SampleReverse( oMixture, dSchedule, numSamples: int, seedNum: int ):
    oRng = np.random.default_rng(seedNum)
    numSteps = len(dSchedule['Beta'])
    mPath = np.empty((numSteps + 1, numSamples))
    mPath[0] = oRng.standard_normal(numSamples)
    for pathIdx, stepIdx in enumerate(range(numSteps, 0, -1), start = 1):
        vCurrent = mPath[pathIdx - 1]
        vClean = oMixture.Denoise(vCurrent, dSchedule['AlphaBar'][stepIdx])
        vMean = dSchedule['CoefClean'][stepIdx - 1] * vClean + dSchedule['CoefNoisy'][stepIdx - 1] * vCurrent
        if stepIdx > 1:
            mPath[pathIdx] = vMean + math.sqrt(dSchedule['Variance'][stepIdx - 1]) * oRng.standard_normal(numSamples)
        else:
            mPath[pathIdx] = vMean
    return mPath

In [ ]:
# Reverse Trajectories and Their Ensemble Mean

mReversePath = SampleReverse(oMixture, dSchedule, numSamples, seedNum + 2)
vGenerated = mReversePath[-1]
vEnsembleMean = mReversePath.mean(axis = 1)
vReverseProgress = np.linspace(0, 1, numDiffSteps + 1)
numShown = min(numTracks, numSamples)

hF, hA = plt.subplots(figsize = (11, 5))
for compIdx, (compMean, compVar) in enumerate(zip(oMixture.vMeans, oMixture.vVars)):
    hA.axhspan(compMean - 2 * math.sqrt(compVar), compMean + 2 * math.sqrt(compVar), color = mColors[compIdx], alpha = 0.08)
    hA.axhline(compMean, color = mColors[compIdx], linestyle = ':', linewidth = 1)
for trackIdx in range(numShown):
    compIdx = int(np.argmin(np.abs(vGenerated[trackIdx] - oMixture.vMeans)))
    hA.plot(vReverseProgress, mReversePath[:, trackIdx], color = mColors[compIdx], alpha = 0.45, linewidth = 0.8, label = f'Trajectories ({numShown} shown)' if trackIdx == 0 else '_nolegend_')
hA.plot(vReverseProgress, vEnsembleMean, color = meanColor, linewidth = 3, label = f'Ensemble mean ({numSamples:,} samples)')
trackLimit = max(axisLimit, 1.05 * np.abs(mReversePath[:, :numShown]).max())
hA.set(xlim = (0, 1), ylim = (-trackLimit, trackLimit), xlabel = 'Reverse Progress (Noise to Data)', ylabel = r'State $y_t$', title = 'Individual Samples Separate; Their Average Does Not')
hA.set_xticks((0, 0.25, 0.5, 0.75, 1), ('0 (Noise)', '0.25', '0.50', '0.75', '1 (Data)'))
hA.legend(loc = 'upper left')
FormatAxes(hA)
ShowFigure(hF, '05ReverseTrajectories')

print(f'Sample mean: {vGenerated.mean():.4f} | Fraction above zero: {(vGenerated > 0).mean():.3f}')

### Compare Distributions, Not Only Averages

The histogram should populate both groups and approximately follow the known data density. A correct average alone would not establish that.

This is _Sampling_, not an optimization procedure that searches for a density maximum. Removing reverse noise does not turn this DDPM update into a MAP solver or automatically give a valid deterministic sampler.

* <font color='brown'>(**#**)</font> A VAE can also generate multiple modes through different latent samples. Its decoder mean averages only the uncertainty remaining after conditioning on its latent and any input condition.
* <font color='blue'>(**!**)</font> Reduce `numDiffSteps` and rerun the schedule and reverse examples. Inspect group width and frequency, not just the mean.
* <font color='red'>(**?**)</font> What would this histogram look like if we replaced every generated sample by their ensemble average?

In [ ]:
# Generated Samples Against the Known Density

hF, hA = plt.subplots(figsize = (9, 4.5))
hA.hist(vGenerated, bins = np.linspace(-axisLimit, axisLimit, 100), density = True, color = mColors[0], alpha = 0.55, label = 'DDPM samples')
hA.plot(vGrid, oMixture.Density(vGrid), color = densityColor, linewidth = 2.5, label = 'Data density')
hA.axvline(vGenerated.mean(), color = meanColor, linestyle = '--', linewidth = 2, label = f'Sample mean = {vGenerated.mean():.3f}')
hA.set(xlim = (-axisLimit, axisLimit), xlabel = r'$y_0$', ylabel = 'Density', title = 'A Mean Near Zero, but Samples in Both Groups')
hA.legend(loc = 'upper center')
FormatAxes(hA)
ShowFigure(hF, '06GeneratedDistribution')

print('Group         Fraction     Mean      Std')
vNearest = np.abs(vGenerated[:, None] - oMixture.vMeans).argmin(axis = 1)
for compIdx, compMean in enumerate(oMixture.vMeans):
    vGroup = vGenerated[vNearest == compIdx]
    if vGroup.size:
        print(f'{compMean:>6.1f}        {vGroup.size / numSamples:.3f}     {vGroup.mean():>6.3f}    {vGroup.std():.3f}')

## Conditioning Changes the Distribution

Let an input $x$ change the probability of each group:

$$ p(y_0 \mid x) = \sum_k \pi_k(x)\,\mathcal{N}(y_0;\mu_k,\sigma_k^2), \qquad D^*(y_t,t,x) = \mathbb{E}[y_0 \mid y_t,t,x] $$

The condition supplies prior information; the noisy state still matters. Here we keep the group locations fixed and change only their weights. Each panel uses the same random seed and schedule, but a different conditional denoiser.

The condition changes how often each group appears. It does not require every output to equal $\mathbb{E}[y_0\mid x]$.

* <font color='red'>(**?**)</font> If the right group has probability $0.9$, should all generated samples come from it?
* <font color='blue'>(**!**)</font> Change `tuRightProbs` and compare the requested probabilities with the generated fractions.

In [ ]:
# Conditional Generation with Different Group Probabilities

tuRightProbs = (0.1, 0.5, 0.9)
numConditionSamples = min(numSamples, 4000)
lConditionSamples = []

hF, vHA = plt.subplots(1, len(tuRightProbs), figsize = (14, 4.5), sharex = True, sharey = True, squeeze = False)
for rightProb, hA in zip(tuRightProbs, vHA[0]):
    oConditional = GaussianMixture1D(tuMeans, tuStds, (1 - rightProb, rightProb))
    vConditional = SampleReverse(oConditional, dSchedule, numConditionSamples, seedNum + 3)[-1].copy()
    lConditionSamples.append(vConditional)
    conditionalMean = float(oConditional.vWeights @ oConditional.vMeans)
    rightFraction = (np.abs(vConditional[:, None] - oConditional.vMeans).argmin(axis = 1) == 1).mean()
    hA.hist(vConditional, bins = np.linspace(-axisLimit, axisLimit, 100), density = True, color = mColors[0], alpha = 0.5)
    hA.plot(vGrid, oConditional.Density(vGrid), color = densityColor, linewidth = 2.5, label = 'Conditional density')
    hA.axvline(conditionalMean, color = meanColor, linestyle = '--', linewidth = 1.5, label = 'Conditional mean')
    hA.set(xlim = (-axisLimit, axisLimit), xlabel = r'$y_0$', title = f'Right Group: Target {rightProb:.0%} | Generated {rightFraction:.1%}')
    FormatAxes(hA)
vHA[0, 0].set_ylabel('Density')
vHA[0, 1].legend(loc = 'upper center')
ShowFigure(hF, '07Conditioning')

## Different Targets, Different Emphasis

Write $y_t=a_ty_0+b_t\epsilon$, with $a_t=\sqrt{\bar\alpha_t}$ and $b_t=\sqrt{1-\bar\alpha_t}$. We may predict the clean value, the noise, or the _Velocity_ $v_t=a_t\epsilon-b_ty_0$.

For $0<\bar\alpha_t<1$, all three targets can recover the clean value. Their squared errors assign different weights to the same clean-prediction error:

| Target | Recover $\hat y_0$ | Weight on $(\hat y_0-y_0)^2$ |
|--------|---------------------|--------------------------------|
| Noise $\hat\epsilon$ | $(y_t-b_t\hat\epsilon)/a_t$ | $\mathrm{SNR}(t)$ |
| Clean $\hat y_0$ | $\hat y_0$ | $1$ |
| Velocity $\hat v$ | $a_ty_t-b_t\hat v$ | $1+\mathrm{SNR}(t)$ |

$$ \mathrm{SNR}(t)=\frac{\bar\alpha_t}{1-\bar\alpha_t} $$

Thus, choosing a target also chooses an implicit emphasis across noise levels. Noise prediction strongly weights clean error at high signal levels; velocity retains a unit-weight floor as the signal vanishes.

We can also multiply a loss by a positive time-dependent weight. That changes which errors influence training most, not the forward noise schedule. Sampling does not minimize this training loss at each step.

* <font color='brown'>(**#**)</font> These equivalences use MSE and the corresponding clean estimate, not SmoothL1. They do not guarantee identical finite-network training or make one target universally best. This notebook's sampler uses the analytic clean denoiser.
* <font color='red'>(**?**)</font> Which target makes a fixed clean-prediction error least costly near pure noise?

In [ ]:
# Implicit MSE Weights Along the Diffusion Path

vTime = np.arange(1, numDiffSteps) / numDiffSteps
vSignal = dSchedule['AlphaBar'][1:-1]
vSnr = vSignal / (1 - vSignal)

hF, hA = plt.subplots(figsize = (9, 4.5))
hA.semilogy(vTime, vSnr, color = densityColor, linewidth = 2.5, label = 'Noise: SNR')
hA.semilogy(vTime, np.ones_like(vTime), color = meanColor, linestyle = '--', linewidth = 2, label = 'Clean: 1')
hA.semilogy(vTime, 1 + vSnr, color = mColors[1], linewidth = 2.5, label = 'Velocity: 1 + SNR')
hA.set(xlim = (0, 1), xlabel = r'Forward Time $t / T$ (Data to Noise)', ylabel = 'Weight on Squared Clean Error', title = 'Same Clean Error, Different Training Emphasis')
hA.legend(loc = 'lower left')
FormatAxes(hA)
ShowFigure(hF, '08TargetWeights')

## Explore the Model

* <font color='blue'>(**!**)</font> Increase `tuStds` until the two groups overlap. Compare the data density, conditional-mean curves, and generated distribution.
* <font color='blue'>(**!**)</font> Use unequal `tuWeights` and rerun the one-dimensional examples. Does the ensemble mean still remain near zero? Which captions describe only the balanced default?
* <font color='red'>(**?**)</font> What information comes from the condition, what comes from the noisy state, and what creates diversity across trajectories?
* <font color='green'>(**@**)</font> Replace the analytic denoiser with a small trained network. Compare its predictions with `Denoise()` before comparing generated samples.

The central distinction: a denoiser returns a _Conditional Mean_, while a sampler generates a _Distribution_. Averaging the final samples is a separate operation.